#🩺 TAE-IA

# **Inspector de Dispositivos Médicos — SATDM**
#**Elaboro: Juan Carlos Aquino Hernández**
## App de producción: visión y voz para el Sistema Auditable de Trazabilidad de Dispositivos Médicos

**Objetivo:** versión de producción del pipeline *Inspector de Dispositivos* (visión + voz, manos-libres) aplicada al caso de uso real
del **SATDM** (Sistema Auditable de Trazabilidad de Dispositivos Médicos para la Mejora de la Seguridad del Paciente
en la Red de Salud Pública de Nayarit): un técnico o enfermero toma una foto (o video) de un dispositivo médico (bomba de
infusión, ventilador, monitor, etc.), opcionalmente graba una nota de voz, y el sistema:

1. **Valida** que la imagen corresponda efectivamente a un dispositivo médico (*gatekeeper*), rechazando entradas ajenas al dominio.
2. **Detecta y localiza** el dispositivo en la imagen o video mediante YOLO (especializado, con fallback a nano).
3. **Identifica el tipo de dispositivo** mediante clasificación zero-shot (BiomedCLIP).
4. **Evalúa su estado físico/de seguridad** (funcional, dañado, etiqueta ilegible, corrosión, etc.), también zero-shot.
5. **Genera un reporte** (BLIP + Whisper para incorporar la nota verbal del operador).
6. **Registra la inspección** en una bitácora auditable append-only con **cadena de hashes** (CSV `inspecciones_log.csv`
   + `chain_state.json`), ligando evidencia, timestamp e ID de dispositivo — el artefacto de trazabilidad que alimenta al SATDM.

> Esta app extiende el prototipo `L12_inspector_dispositivos_SATDM` (YOLOv8n, Whisper Small, BLIP-large, BiomedCLIP) con:
> detección en video, taxonomías y plantillas propias, funciones de auditoría reforzadas (hash encadenado y directorio de
> evidencia), y una interfaz completa en Gradio lista para campo.

# **Módulos del Sistema**

| Módulo | Acción que realiza | Objetivo |
| :--- | :--- | :--- |
| **1. Setup** | Instala/verifica dependencias (transformers, ultralytics, open_clip, gradio, librosa, opencv) | Preparar el entorno de ejecución |
| **2. Configuración** | Monta Google Drive, define rutas, constantes globales y el esquema de la bitácora, fija semillas | Establecer un entorno reproducible y trazable |
| **3. Modelos** | Carga YOLO (especializado o fallback nano), BiomedCLIP, BLIP y Whisper | Disponer de las piezas del sistema |
| **4. Taxonomías y plantillas** | Define catálogos de tipo/estado de dispositivo y plantillas de reporte | Estandarizar la clasificación y el reporte |
| **5. Utilitarias y auditoría** | Funciones de hashing, escritura append-only y cadena de integridad de la bitácora | Garantizar trazabilidad auditable |
| **6. Inferencia y detección** | Detección optimizada con YOLO | Localizar el dispositivo en la entrada |
| **7. Inspección en imágenes** | Pipeline completo: gatekeeper + clasificación + reporte para fotos | Producir una inspección por imagen |
| **8. Inspección en video** | Extiende el pipeline a clips de video | Cubrir el caso de uso de inspección en video |
| **9. Interfaz Gradio** | App completa con bitácora visible, manos-libres | Uso en campo por personal de salud |


> **Un solo run, sin pasos previos**
>
> Esta app usa YOLOv8 **genérico (COCO, sin fine-tuning)** solo como detector/localizador de objetos en la imagen; la identificación real del tipo de dispositivo y su estado corre por BiomedCLIP (zero-shot) y la descripción por BLIP. No hace falta entrenar nada ni cargar pesos personalizados: todos los modelos se descargan automáticamente desde Hugging Face / Ultralytics la primera vez que se ejecuta esta celda en adelante.

<div style="border: 1px solid #e2e8f0; border-radius: 8px; padding: 20px; background-color: #ffffff; box-shadow: 0 2px 4px rgba(0,0,0,0.02); margin: 15px 0;">
    <div style="display: flex; justify-content: space-between; align-items: center; border-bottom: 2px solid #0284c7; padding-bottom: 10px; margin-bottom: 12px;">
        <h2 style="color: #0f172a; margin: 0; font-size: 20px; border: none;">
            ⚙️ 1. Setup
        </h2>
        <span style="background-color: #e0f2fe; color: #0369a1; padding: 4px 10px; border-radius: 20px; font-size: 12px; font-weight: bold;">
            Fase: Preparación
        </span>
    </div>
    <p style="color: #334155; margin: 0 0 10px 0; font-size: 14px;">
        <strong>Objetivo:</strong> Preparar el entorno de ejecución.<br>
        <strong>Descripción:</strong> Instala/verifica dependencias (<code>transformers</code>, <code>ultralytics</code>, <code>open_clip</code>, <code>gradio</code>, <code>librosa</code>, <code>opencv</code>).
    </p>
</div>

In [1]:
# ============================================================
# 1. INSTALACIÓN Y VERIFICACIÓN DE DEPENDENCIAS
# ============================================================
!pip -q install \
    "transformers==4.49.0" \
    "huggingface-hub==0.29.1" \
    "ultralytics>=8.3.0,<9" \
    "open_clip_torch" \
    "gradio" \
    "librosa" \
    "soundfile" \
    "opencv-python-headless"

import sys, subprocess, importlib

print("Python:", sys.version.split()[0])
print("Comprobando paquetes críticos...")

for pkg in ["transformers", "huggingface_hub", "ultralytics", "open_clip"]:
    try:
        mod = importlib.import_module(pkg)
        print(f" | {pkg}: {getattr(mod, '__version__', 'OK')}")
    except Exception as e:
        print(f" | {pkg}: ERROR -> {e}")

Python: 3.13.15
Comprobando paquetes críticos...
 | transformers: 4.49.0
 | huggingface_hub: 0.29.1
 | ultralytics: 8.4.161
 | open_clip: 3.3.0


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
from pathlib import Path

PESOS_YOLO = Path(
    "/content/drive/MyDrive/TAE_IA_M6/pesos_medicos_yolov8.pt"
)

print("¿Existe?", PESOS_YOLO.exists())
print("Ruta:", PESOS_YOLO)

if PESOS_YOLO.exists():
    print(f"OK - pesos encontrados: {PESOS_YOLO.stat().st_size / 1024**2:.2f} MB")
else:
    print("ERROR: no se encontraron los pesos.")

¿Existe? True
Ruta: /content/drive/MyDrive/TAE_IA_M6/pesos_medicos_yolov8.pt
OK - pesos encontrados: 5.95 MB


<div style="border: 1px solid #e2e8f0; border-radius: 8px; padding: 20px; background-color: #ffffff; box-shadow: 0 2px 4px rgba(0,0,0,0.02); margin: 15px 0;">
    <div style="display: flex; justify-content: space-between; align-items: center; border-bottom: 2px solid #0284c7; padding-bottom: 10px; margin-bottom: 12px;">
        <h2 style="color: #0f172a; margin: 0; font-size: 20px; border: none;">
            🔧 2. Configuración
        </h2>
        <span style="background-color: #e0f2fe; color: #0369a1; padding: 4px 10px; border-radius: 20px; font-size: 12px; font-weight: bold;">
            Fase: Reproducibilidad
        </span>
    </div>
    <p style="color: #334155; margin: 0 0 10px 0; font-size: 14px;">
        <strong>Objetivo:</strong> Establecer un entorno reproducible y trazable.<br>
        <strong>Descripción:</strong> Monta Google Drive, define rutas, constantes globales y el esquema de la bitácora, fija semillas.
    </p>
</div>

In [5]:
# ============================================================
# 2. CONFIGURACIÓN, RUTAS Y CONSTANTES GLOBALES
# ============================================================
import os, io, csv, json, time, uuid, random, hashlib, math, traceback
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch
import cv2
from PIL import Image, ImageDraw, ImageFont

try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = "/content/drive/MyDrive/TAE_IA_M6"
except Exception:
    DRIVE_ROOT = "/content/TAE_IA_M6"

OUTPUT_DIR = os.path.join(DRIVE_ROOT, "SATDM_output")
INPUT_DIR = os.path.join(DRIVE_ROOT, "inputs")
LEDGER_DIR = os.path.join(DRIVE_ROOT, "SATDM_ledger")
EVIDENCE_DIR = os.path.join(LEDGER_DIR, "evidence")
MODEL_CACHE = "/content/models"

for d in [OUTPUT_DIR, INPUT_DIR, LEDGER_DIR, EVIDENCE_DIR, MODEL_CACHE]:
    os.makedirs(d, exist_ok=True)

os.environ["HF_HOME"] = MODEL_CACHE
os.environ["TORCH_HOME"] = MODEL_CACHE
os.environ["YOLO_CONFIG_DIR"] = MODEL_CACHE
os.environ["TTS_HOME"] = MODEL_CACHE

LEDGER_PATH = os.path.join(LEDGER_DIR, "inspecciones_log.csv")
CHAIN_STATE_PATH = os.path.join(LEDGER_DIR, "chain_state.json")

LEDGER_FIELDS = [
    "inspection_id", "timestamp_utc", "media_type", "device_id",
    "evidence_filename", "evidence_sha256", "previous_record_hash", "record_hash",
    "gatekeeper_passed", "gatekeeper_score", "gatekeeper_reason",
    "tipo_detectado", "tipo_confianza", "tipo_entropia",
    "estado_detectado", "estado_confianza", "estado_entropia",
    "operator_note", "review_required", "model_version",
    "evidence_path", "report_path", "audio_report_path",
    "frame_start", "frame_end", "track_id",
    # --- fusionado desde M6_L24 (CLAP): evidencia acústica del propio dispositivo ---
    "audio_tag_detectado", "audio_tag_confianza", "audio_tiene_voz",
    # --- texto del reporte narrativo, visible directamente en la bitácora ---
    "resumen_reporte"
]

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

ENABLE_TTS = False

# ============================================================
# MODELO YOLO MÉDICO PERSONALIZADO
# Integrado desde app_inspector_dispositivos_SATDM
# ============================================================
# Si existe el peso entrenado en Drive, se utiliza automáticamente.
# Si no existe, la aplicación conserva un fallback funcional con yolov8n.pt.
MEDICAL_YOLO_WEIGHTS = os.environ.get(
    "SATDM_YOLO_WEIGHTS",
    os.path.join(DRIVE_ROOT, "pesos_medicos_yolov8.pt")
)

YOLO_USE_MEDICAL = True
YOLO_MODEL_SOURCE = "yolov8n.pt (COCO, fallback)"



MAX_SIDE = 1280
MIN_SIDE = 64
YOLO_CONF = 0.25
MEDICAL_THRESHOLD = 0.55
AMBIGUITY_ENTROPY = 0.85
VIDEO_FRAME_STRIDE = 15
MIN_TRACK_OBSERVATIONS = 2

IMAGE_EXTS = (".jpg", ".jpeg", ".png", ".webp", ".bmp")
VIDEO_EXTS = (".mp4", ".avi", ".mov", ".mkv", ".webm")
AUDIO_EXTS = (".wav", ".mp3", ".m4a", ".flac", ".ogg")

REFERENCE_VOICE = os.path.join(INPUT_DIR, 'reference_voice.wav')

def vram(tag=""):
    if DEVICE != "cuda":
        print(f"[VRAM] CPU — {tag}")
        return
    used = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"[VRAM] {used:.2f}/{total:.1f} GB — {tag}")

print("Dispositivo configurado:", DEVICE)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dispositivo configurado: cpu


In [6]:
# ============================================================
# 2b. MIGRACIÓN AUTOMÁTICA DEL ESQUEMA DE LA BITÁCORA
# Se ejecuta una sola vez, automáticamente, si detecta un encabezado
# desactualizado (por ejemplo, un CSV creado antes de agregar los
# campos de CLAP o resumen_reporte a LEDGER_FIELDS).
# ============================================================
def _migrar_bitacora_si_necesario():
    if not os.path.exists(LEDGER_PATH):
        return

    with open(LEDGER_PATH, encoding="utf-8", newline="") as f:
        rows = list(csv.reader(f))

    if not rows:
        return

    old_header = rows[0]
    if old_header == LEDGER_FIELDS:
        return  # el esquema ya está al día, no hacer nada

    data_rows = rows[1:]
    print(f"[Migración] Encabezado desactualizado: {len(old_header)} columnas "
          f"(se esperaban {len(LEDGER_FIELDS)}).")

    new_cols = [c for c in LEDGER_FIELDS if c not in old_header]
    records = []
    for row in data_rows:
        if len(row) < len(old_header):
            row = row + [""] * (len(old_header) - len(row))
        extra, row = row[len(old_header):], row[:len(old_header)]
        record = dict(zip(old_header, row))
        for col, val in zip(new_cols, extra):
            record[col] = val
        records.append(record)

    backup_path = LEDGER_PATH + f".bak_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S')}"
    os.rename(LEDGER_PATH, backup_path)
    print(f"[Migración] Respaldo guardado en: {backup_path}")

    with open(LEDGER_PATH, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=LEDGER_FIELDS)
        writer.writeheader()
        for r in records:
            writer.writerow({k: r.get(k, "") for k in LEDGER_FIELDS})

    print(f"[Migración] {len(records)} registros migrados a {len(LEDGER_FIELDS)} columnas.")

_migrar_bitacora_si_necesario()


<div style="border: 1px solid #e2e8f0; border-radius: 8px; padding: 20px; background-color: #ffffff; box-shadow: 0 2px 4px rgba(0,0,0,0.02); margin: 15px 0;">
    <div style="display: flex; justify-content: space-between; align-items: center; border-bottom: 2px solid #0284c7; padding-bottom: 10px; margin-bottom: 12px;">
        <h2 style="color: #0f172a; margin: 0; font-size: 20px; border: none;">
            🧠 3. Modelos
        </h2>
        <span style="background-color: #e0f2fe; color: #0369a1; padding: 4px 10px; border-radius: 20px; font-size: 12px; font-weight: bold;">
            Fase: Carga de IA
        </span>
    </div>
    <p style="color: #334155; margin: 0 0 10px 0; font-size: 14px;">
        <strong>Objetivo:</strong> Disponer de las piezas del sistema.<br>
        <strong>Descripción:</strong> Carga YOLO (especializado o fallback nano), BiomedCLIP, BLIP y Whisper.
    </p>
</div>

In [7]:
# ============================================================
# 3. CARGA DE MODELOS — SATDM INTEGRADO
# YOLO Médico + BiomedCLIP + BLIP + Whisper + CLAP
# ============================================================
from ultralytics import YOLO
import open_clip
from transformers import (
    BlipProcessor, BlipForConditionalGeneration,
    WhisperProcessor, WhisperForConditionalGeneration
)

# ------------------------------------------------------------
# 3a. YOLO
# ------------------------------------------------------------
YOLO_IS_MEDICAL = bool(YOLO_USE_MEDICAL and os.path.exists(MEDICAL_YOLO_WEIGHTS))

if YOLO_IS_MEDICAL:
    print(f"\nCargando YOLO especializado para dispositivos médicos:")
    print(f"  {MEDICAL_YOLO_WEIGHTS}")
    yolo = YOLO(MEDICAL_YOLO_WEIGHTS)
    YOLO_MODEL_SOURCE = os.path.basename(MEDICAL_YOLO_WEIGHTS)
else:
    print("\n⚠️ Pesos médicos no encontrados.")
    print(f"   Ruta esperada: {MEDICAL_YOLO_WEIGHTS}")
    print("   Se utilizará yolov8n.pt como fallback para localización.")
    yolo = YOLO("yolov8n.pt")
    YOLO_MODEL_SOURCE = "yolov8n.pt (COCO, fallback)"

print("YOLO:", YOLO_MODEL_SOURCE)
try:
    print("Clases YOLO:", yolo.names)
except Exception:
    pass

# ------------------------------------------------------------
# 3b. BiomedCLIP
# ------------------------------------------------------------
print("\nCargando BiomedCLIP...")
BIOMED_ID = "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224"
try:
    biomed_model, _, biomed_preprocess = open_clip.create_model_and_transforms(
        BIOMED_ID, pretrained=BIOMED_ID, cache_dir=MODEL_CACHE
    )
except TypeError:
    biomed_model, _, biomed_preprocess = open_clip.create_model_and_transforms(
        BIOMED_ID, pretrained=BIOMED_ID
    )

biomed_tokenizer = open_clip.get_tokenizer(BIOMED_ID)
biomed_model = biomed_model.to(DEVICE).eval()

# ------------------------------------------------------------
# 3c. BLIP
# ------------------------------------------------------------
print("Cargando BLIP...")
blip_proc = BlipProcessor.from_pretrained(
    "Salesforce/blip-image-captioning-large",
    cache_dir=MODEL_CACHE
)
blip_model = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-large",
    cache_dir=MODEL_CACHE,
    torch_dtype=DTYPE
).to(DEVICE).eval()

# ------------------------------------------------------------
# 3d. Whisper
# ------------------------------------------------------------
print("Cargando Whisper...")
whisper_proc = WhisperProcessor.from_pretrained(
    "openai/whisper-small",
    cache_dir=MODEL_CACHE
)
whisper_model = WhisperForConditionalGeneration.from_pretrained(
    "openai/whisper-small",
    cache_dir=MODEL_CACHE,
    torch_dtype=DTYPE
).to(DEVICE).eval()

# ------------------------------------------------------------
# 3e. XTTS opcional
# ------------------------------------------------------------
tts = None
if ENABLE_TTS:
    try:
        from TTS.api import TTS
        tts = TTS(
            "tts_models/multilingual/multi-dataset/xtts_v2"
        ).to(DEVICE)
        print("XTTS habilitado.")
    except Exception as e:
        print("TTS no activado o no disponible:", e)

# ------------------------------------------------------------
# 3f. CLAP — clasificación acústica zero-shot
# ------------------------------------------------------------
from transformers import pipeline as hf_pipeline

print("\nCargando CLAP para análisis acústico...")
try:
    clap_tagger = hf_pipeline(
        "zero-shot-audio-classification",
        model="laion/clap-htsat-unfused",
        device=0 if DEVICE == "cuda" else -1
    )
    CLAP_AVAILABLE = True
    print("CLAP disponible.")
except Exception as e:
    clap_tagger = None
    CLAP_AVAILABLE = False
    print("⚠️ CLAP no pudo cargarse:", e)
    print("La inspección continuará usando Whisper cuando exista audio.")

vram("Modelos SATDM integrados cargados")



Cargando YOLO especializado para dispositivos médicos:
  /content/drive/MyDrive/TAE_IA_M6/pesos_medicos_yolov8.pt


YOLO: pesos_medicos_yolov8.pt
Clases YOLO: {0: 'Bed', 1: 'IV Stand', 2: 'Monitor', 3: 'Wheelchair'}

Cargando BiomedCLIP...


open_clip_config.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

open_clip_pytorch_model.bin:   0%|          | 0.00/784M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

Device set to use cpu


CLAP disponible.
[VRAM] CPU — Modelos SATDM integrados cargados


In [8]:
print(yolo.names)

{0: 'Bed', 1: 'IV Stand', 2: 'Monitor', 3: 'Wheelchair'}


<div style="border: 1px solid #e2e8f0; border-radius: 8px; padding: 20px; background-color: #ffffff; box-shadow: 0 2px 4px rgba(0,0,0,0.02); margin: 15px 0;">
    <div style="display: flex; justify-content: space-between; align-items: center; border-bottom: 2px solid #0284c7; padding-bottom: 10px; margin-bottom: 12px;">
        <h2 style="color: #0f172a; margin: 0; font-size: 20px; border: none;">
            📋 4. Taxonomías y plantillas
        </h2>
        <span style="background-color: #e0f2fe; color: #0369a1; padding: 4px 10px; border-radius: 20px; font-size: 12px; font-weight: bold;">
            Fase: Estandarización
        </span>
    </div>
    <p style="color: #334155; margin: 0 0 10px 0; font-size: 14px;">
        <strong>Objetivo:</strong> Estandarizar la clasificación y el reporte.<br>
        <strong>Descripción:</strong> Define catálogos de tipo/estado de dispositivo y plantillas de reporte.
    </p>
</div>

In [9]:
# ============================================================
# 4. TAXONOMÍAS Y PLANTILLAS SATDM
# ============================================================

DEVICE_TYPE_TEMPLATES = {

    "autoclave": [
        "a medical autoclave sterilizer",
        "a hospital steam sterilization autoclave",
        "a clinical autoclave machine",
        "medical equipment used for instrument sterilization",
    ],

    "bomba de infusión": [
        "a medical infusion pump",
        "a hospital IV infusion pump with tubing",
        "a clinical infusion pump",
        "an intravenous infusion pump",
    ],

    "cama de hospital": [
        "a hospital bed with rails and controls",
        "a clinical adjustable hospital bed",
        "a medical patient bed",
        "a hospital patient bed",
    ],

    "desfibrilador": [
        "a medical defibrillator",
        "a hospital defibrillator machine",
        "an automated external defibrillator",
        "a clinical cardiac defibrillator",
    ],

    "electrocardiógrafo": [
        "an electrocardiograph ECG machine",
        "a hospital ECG device with cables",
        "a clinical electrocardiogram machine",
        "a medical ECG recorder",
    ],

    "equipo de resonancia magnética": [
        "a medical magnetic resonance imaging MRI machine",
        "a hospital MRI scanner",
        "a clinical magnetic resonance imaging system",
        "an MRI medical imaging machine",
    ],

    "monitorización de signos vitales": [
        "a patient vital signs monitor with a screen",
        "a hospital bedside vital signs monitor",
        "a clinical multiparameter patient monitor",
        "medical equipment for monitoring vital signs",
    ],

    "oxímetro de pulso": [
        "a pulse oximeter medical device",
        "a fingertip pulse oximeter",
        "a medical oxygen saturation monitor",
        "a clinical pulse oximetry device",
    ],

    "radiografía médica": [
        "a medical X-ray machine",
        "a hospital X-ray radiography system",
        "a clinical medical radiography machine",
        "a diagnostic X-ray imaging device",
        "a portable medical X-ray machine",
    ],

    "silla de ruedas": [
        "a medical wheelchair",
        "a hospital wheelchair",
        "a clinical transport wheelchair",
        "a patient mobility wheelchair",
    ],

    "tanque de oxígeno": [
        "a medical oxygen cylinder",
        "a hospital oxygen supply tank",
        "a clinical oxygen cylinder",
        "a medical oxygen tank",
    ],

    "ventilador médico": [
        "a medical ventilator",
        "a mechanical ventilator for breathing support",
        "a hospital ventilator with tubes and display",
        "an ICU mechanical ventilator",
    ],
}


# ============================================================
# 4a. TAXONOMÍA DEL ESTADO DEL DISPOSITIVO
# ============================================================

STATUS_TEMPLATES = {

    "funcional / buen estado": [
        "a medical device in good working condition",
        "clean and well maintained medical equipment",
        "a functional medical device in good condition",
    ],

    "dañado / daño visible": [
        "a medical device with visible physical damage",
        "broken or cracked hospital equipment",
        "medical equipment with visible damage",
    ],

    "etiqueta o número de serie ilegible": [
        "a medical device with a faded or illegible label",
        "hospital equipment with an unreadable serial number",
        "medical equipment with an illegible identification label",
    ],

    "sucio o contaminado visible": [
        "a medical device that looks dirty or stained",
        "hospital equipment with visible dirt or residue",
        "medical equipment with visible contamination",
    ],

    "corrosión u oxidación visible": [
        "a medical device with visible rust or corrosion",
        "hospital equipment showing corrosion damage",
        "medical equipment with visible oxidation",
    ],

    "cables o componentes expuestos": [
        "a medical device with exposed wires or cables",
        "hospital equipment with exposed components",
        "medical equipment with exposed electrical components",
    ],

    "estado no concluyente": [
        "a medical device whose physical condition cannot be determined",
        "medical equipment with inconclusive physical condition",
    ],
}


# ============================================================
# 4b. GATEKEEPER
# ============================================================

GATEKEEPER_TEMPLATES = [
    "a medical device",
    "hospital medical equipment",
    "a clinical medical instrument",
    "medical imaging equipment",
    "a non-medical everyday object",
    "a person",
    "an animal",
    "food",
    "a landscape",
    "a document",
]

MEDICAL_GATE_LABELS = {
    "a medical device",
    "hospital medical equipment",
    "a clinical medical instrument",
    "medical imaging equipment",
}

# ------------------------------------------------------------
# Confianza asignada cuando el tipo/estado se decide por voz del
# operador en vez de por clasificación visual. NO se deja en 1.0
# para evitar que una mención de voz suprima artificialmente la
# revisión manual (review_required) cuando corresponda.
# ------------------------------------------------------------
VOICE_CONF_OVERRIDE = 0.85


# ============================================================
# 4c. RECONOCIMIENTO DEL DISPOSITIVO POR VOZ DEL OPERADOR
# ============================================================

VOICE_DEVICE_KEYWORDS = {

    # --------------------------------------------------------
    # AUToclave
    # --------------------------------------------------------
    "autoclave": "autoclave",
    "esterilizador": "autoclave",
    "esterilizador medico": "autoclave",
    "esterilizador médico": "autoclave",

    # --------------------------------------------------------
    # BOMBA DE INFUSIÓN
    # --------------------------------------------------------
    "bomba de infusion": "bomba de infusión",
    "bomba de infusión": "bomba de infusión",
    "bomba de suero": "bomba de infusión",
    "infusora": "bomba de infusión",
    "bomba intravenosa": "bomba de infusión",

    # --------------------------------------------------------
    # CAMA DE HOSPITAL
    # --------------------------------------------------------
    "cama de hospital": "cama de hospital",
    "cama hospitalaria": "cama de hospital",
    "cama clinica": "cama de hospital",
    "cama clínica": "cama de hospital",
    "cama medica": "cama de hospital",
    "cama médica": "cama de hospital",

    # --------------------------------------------------------
    # DESFIBRILADOR
    # --------------------------------------------------------
    "desfibrilador": "desfibrilador",
    "desfibrilador medico": "desfibrilador",
    "desfibrilador médico": "desfibrilador",
    "dea": "desfibrilador",

    # --------------------------------------------------------
    # ELECTROCARDIÓGRAFO
    # --------------------------------------------------------
    "electrocardiografo": "electrocardiógrafo",
    "electrocardiógrafo": "electrocardiógrafo",
    "electrocardiograma": "electrocardiógrafo",
    "ecg": "electrocardiógrafo",
    "ekg": "electrocardiógrafo",

    # --------------------------------------------------------
    # RESONANCIA MAGNÉTICA
    # --------------------------------------------------------
    "resonancia magnetica": "equipo de resonancia magnética",
    "resonancia magnética": "equipo de resonancia magnética",
    "resonancia": "equipo de resonancia magnética",
    "rm": "equipo de resonancia magnética",
    "mri": "equipo de resonancia magnética",

    # --------------------------------------------------------
    # MONITORIZACIÓN DE SIGNOS VITALES
    # --------------------------------------------------------
    "monitor de signos vitales": "monitorización de signos vitales",
    "monitor de signos": "monitorización de signos vitales",
    "monitorizacion de signos vitales": "monitorización de signos vitales",
    "monitorización de signos vitales": "monitorización de signos vitales",
    "monitor multiparametrico": "monitorización de signos vitales",
    "monitor multiparamétrico": "monitorización de signos vitales",

    # --------------------------------------------------------
    # OXÍMETRO
    # --------------------------------------------------------
    "oximetro": "oxímetro de pulso",
    "oxímetro": "oxímetro de pulso",
    "oximetro de pulso": "oxímetro de pulso",
    "oxímetro de pulso": "oxímetro de pulso",
    "oximetria": "oxímetro de pulso",
    "oximetría": "oxímetro de pulso",

    # --------------------------------------------------------
    # RADIOGRAFÍA MÉDICA / RAYOS X
    # --------------------------------------------------------
    "rayos x": "radiografía médica",
    "rayos-x": "radiografía médica",
    "rayos x medico": "radiografía médica",
    "rayos x médico": "radiografía médica",
    "equipo de rayos x": "radiografía médica",
    "equipo de rayos-x": "radiografía médica",
    "equipo de radiografia": "radiografía médica",
    "equipo de radiografía": "radiografía médica",
    "radiografia": "radiografía médica",
    "radiografía": "radiografía médica",
    "radiografia medica": "radiografía médica",
    "radiografía médica": "radiografía médica",
    "maquina de rayos x": "radiografía médica",
    "máquina de rayos x": "radiografía médica",
    "x ray": "radiografía médica",
    "x-ray": "radiografía médica",

    # --------------------------------------------------------
    # SILLA DE RUEDAS
    # --------------------------------------------------------
    "silla de ruedas": "silla de ruedas",
    "silla ruedas": "silla de ruedas",
    "silla hospitalaria": "silla de ruedas",

    # --------------------------------------------------------
    # TANQUE DE OXÍGENO
    # --------------------------------------------------------
    "tanque de oxigeno": "tanque de oxígeno",
    "tanque de oxígeno": "tanque de oxígeno",
    "cilindro de oxigeno": "tanque de oxígeno",
    "cilindro de oxígeno": "tanque de oxígeno",
    "oxigeno": "tanque de oxígeno",
    "oxígeno": "tanque de oxígeno",

    # --------------------------------------------------------
    # VENTILADOR MÉDICO
    # --------------------------------------------------------
    "ventilador medico": "ventilador médico",
    "ventilador médico": "ventilador médico",
    "ventilador mecanico": "ventilador médico",
    "ventilador mecánico": "ventilador médico",
    "respirador": "ventilador médico",
    "ventilador": "ventilador médico",
}


# ============================================================
# 4d. NORMALIZACIÓN Y DETECCIÓN DE DISPOSITIVO POR VOZ
# ============================================================

import re
import unicodedata


def _normalize_text_es(text):
    """
    Normaliza texto español para facilitar coincidencias
    entre Whisper y el vocabulario del SATDM.
    """
    text = str(text or "").lower().strip()

    # Eliminar acentos
    text = unicodedata.normalize("NFD", text)
    text = "".join(
        ch for ch in text
        if unicodedata.category(ch) != "Mn"
    )

    # Normalizar espacios
    text = re.sub(r"\s+", " ", text)

    return text


def detect_device_from_voice(transcript):
    """
    Detecta el dispositivo mencionado explícitamente
    por el operador mediante Whisper.

    IMPORTANTE:
    Esta función NO sustituye la clasificación visual.
    Su resultado se conserva como evidencia independiente.
    """

    normalized_transcript = _normalize_text_es(transcript)

    if not normalized_transcript:
        return {
            "label": None,
            "confidence": 0.0,
            "matched_phrase": None,
            "source": "operator_voice",
        }

    # Normalizar también las palabras clave
    normalized_keywords = []

    for phrase, label in VOICE_DEVICE_KEYWORDS.items():
        normalized_keywords.append(
            (
                _normalize_text_es(phrase),
                label,
                phrase
            )
        )

    # Buscar primero frases largas/específicas
    normalized_keywords.sort(
        key=lambda x: len(x[0]),
        reverse=True
    )

    for normalized_phrase, label, original_phrase in normalized_keywords:

        if normalized_phrase in normalized_transcript:

            return {
                "label": label,
                "confidence": VOICE_CONF_OVERRIDE,
                "matched_phrase": original_phrase,
                "source": "operator_voice",
            }

    return {
        "label": None,
        "confidence": 0.0,
        "matched_phrase": None,
        "source": "operator_voice",
    }


# ============================================================
# 4e. TAXONOMÍA ACÚSTICA CLAP
# ============================================================

AUDIO_EVENT_TEMPLATES = [

    "a medical device alarm beeping",

    "a medical device operating normally",

    "a motor or pump running normally",

    "an abnormal grinding or rattling mechanical noise",

    "a person speaking",

    "background hospital ambient noise",

    "silence",
]


# ------------------------------------------------------------
# CLAP: umbral conservador
# ------------------------------------------------------------
# CLAP se utiliza para eventos acústicos.
# NO debe utilizarse por sí solo para determinar el tipo
# de dispositivo médico.
# ------------------------------------------------------------

AUDIO_CONF_FLOOR = 0.55

<div style="border: 1px solid #e2e8f0; border-radius: 8px; padding: 20px; background-color: #ffffff; box-shadow: 0 2px 4px rgba(0,0,0,0.02); margin: 15px 0;">
    <div style="display: flex; justify-content: space-between; align-items: center; border-bottom: 2px solid #0284c7; padding-bottom: 10px; margin-bottom: 12px;">
        <h2 style="color: #0f172a; margin: 0; font-size: 20px; border: none;">
            🔊 4b. Evidencia acústica (fusionado desde M6_L24)
        </h2>
        <span style="background-color: #e0f2fe; color: #0369a1; padding: 4px 10px; border-radius: 20px; font-size: 12px; font-weight: bold;">
            Fase: Auditoría acústica
        </span>
    </div>
    <p style="color: #334155; margin: 0 0 10px 0; font-size: 14px;">
        <strong>Objetivo:</strong> Escuchar el propio dispositivo, no solo la voz del técnico.<br>
        <strong>Descripción:</strong> Contrato de audio (idéntico al de L24), clasificación de evento acústico con CLAP,
        y combinación con la transcripción de Whisper ya existente en el SATDM.
    </p>
</div>

In [10]:
# ============================================================
# 4c. CONTRATO DE AUDIO Y ANÁLISIS ACÚSTICO (fusionado desde M6_L24)
# ============================================================
import librosa

CLAP_SR = 48000            # medido en L24: fuera de esta tasa, CLAP falla intermitentemente
AUDIO_MAX_SECONDS = 60
AUDIO_MIN_SECONDS = 0.5


def coerce_audio(wav, sr, target_sr):
    """Cualquier entrada de audio -> float32 mono a target_sr, o ValueError.

    Mismo contrato que en L24: llamar SIEMPRE antes de que un modelo vea el audio.
    """
    if wav is None:
        raise ValueError("No se recibió audio.")
    wav = np.asarray(wav)
    if wav.dtype.kind in "iu":                       # int16 tal como lo entrega Gradio
        wav = wav.astype("float32") / np.iinfo(wav.dtype).max
    wav = wav.astype("float32")
    if wav.ndim > 1:                                 # a mono, cualquier layout
        wav = wav.mean(axis=0) if wav.shape[0] < wav.shape[1] else wav.mean(axis=1)
    if wav.size < sr * AUDIO_MIN_SECONDS:
        raise ValueError(f"Clip muy corto ({wav.size/sr:.2f}s).")
    if wav.size > sr * AUDIO_MAX_SECONDS:
        wav = wav[: int(sr * AUDIO_MAX_SECONDS)]     # truncar, no fallar
    if sr != target_sr:
        wav = librosa.resample(wav, orig_sr=sr, target_sr=target_sr)
    return wav.astype("float32"), target_sr


def tag_audio_event(wav, sr):
    """Clasifica eventos acústicos mediante CLAP zero-shot."""
    if not CLAP_AVAILABLE or clap_tagger is None:
        return []
    assert sr == CLAP_SR, f"{sr} != {CLAP_SR} - usa coerce_audio primero"
    result = clap_tagger(wav, candidate_labels=AUDIO_EVENT_TEMPLATES)
    return sorted(
        ((r["label"], float(r["score"])) for r in result),
        key=lambda p: p[1],
        reverse=True
    )


def analyze_audio_note(audio_path):
    """Analiza la nota de audio del técnico / sonido del dispositivo.

    Clasifica el evento acústico (CLAP) y transcribe la voz si la hay (Whisper,
    reutilizando transcribe_audio ya definida en el SATDM). Nunca lanza excepción.

    Devuelve dict: {'audio_tag', 'audio_tag_score', 'has_speech', 'transcript', 'summary'}
    """
    default = {
        "audio_tag": "", "audio_tag_score": 0.0,
        "has_speech": False, "transcript": "",
        "summary": "Sin nota de audio.",
    }
    if not audio_path or not os.path.exists(audio_path):
        return default

    try:
        wav, sr = librosa.load(audio_path, sr=None, mono=True)
        wav_clap, sr_clap = coerce_audio(wav, sr, CLAP_SR)
        tags = tag_audio_event(wav_clap, sr_clap)

        transcript = transcribe_audio(audio_path)
        if tags:
            top_label, top_score = tags[0]
        else:
            top_label, top_score = "", 0.0
        has_speech = bool(transcript.strip())

        evento_txt = top_label if top_score >= AUDIO_CONF_FLOOR else "no_concluyente"
        if not CLAP_AVAILABLE:
            evento_txt = "CLAP_no_disponible"

        summary = f"Evento acústico: {evento_txt} (confianza {top_score:.2f})."
        if has_speech:
            summary += f' Voz detectada: "{transcript}"'

        return {
            "audio_tag": evento_txt,
            "audio_tag_score": top_score,
            "has_speech": has_speech,
            "transcript": transcript,
            "summary": summary,
        }
    except Exception as e:
        return {**default, "summary": f"Error al analizar el audio: {type(e).__name__}: {e}"}


print("Contrato de audio y análisis acústico (CLAP + Whisper) listos.")


Contrato de audio y análisis acústico (CLAP + Whisper) listos.


<div style="border: 1px solid #e2e8f0; border-radius: 8px; padding: 20px; background-color: #ffffff; box-shadow: 0 2px 4px rgba(0,0,0,0.02); margin: 15px 0;">
    <div style="display: flex; justify-content: space-between; align-items: center; border-bottom: 2px solid #0284c7; padding-bottom: 10px; margin-bottom: 12px;">
        <h2 style="color: #0f172a; margin: 0; font-size: 20px; border: none;">
            🛡️ 5. Utilitarias y auditoría
        </h2>
        <span style="background-color: #e0f2fe; color: #0369a1; padding: 4px 10px; border-radius: 20px; font-size: 12px; font-weight: bold;">
            Fase: Trazabilidad
        </span>
    </div>
    <p style="color: #334155; margin: 0 0 10px 0; font-size: 14px;">
        <strong>Objetivo:</strong> Garantizar trazabilidad auditable.<br>
        <strong>Descripción:</strong> Funciones de hashing, escritura append-only y cadena de integridad de la bitácora.
    </p>
</div>

In [11]:
# ============================================================
# 5. FUNCIONES UTILITARIAS Y DE AUDITORÍA
# ============================================================
def utc_now():
    return datetime.now(timezone.utc).isoformat()

def inspection_id():
    return f"SATDM-{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S')}-{uuid.uuid4().hex[:8].upper()}"

def coerce_image(x):
    if x is None:
        raise ValueError("No se recibió una imagen.")
    if isinstance(x, Image.Image):
        img = x.convert("RGB")
    elif isinstance(x, np.ndarray):
        arr = np.asarray(x)
        if arr.ndim == 2:
            img = Image.fromarray(arr.astype(np.uint8)).convert("RGB")
        elif arr.ndim == 3:
            img = Image.fromarray(arr.astype(np.uint8)).convert("RGB")
        else:
            raise ValueError("Dimensiones no soportadas.")
    else:
        raise ValueError("Tipo de imagen no soportado.")

    w, h = img.size
    if min(w, h) < MIN_SIDE:
        raise ValueError(f"Imagen muy pequeña ({w}x{h}). Mínimo {MIN_SIDE}x{MIN_SIDE}.")
    if max(w, h) > MAX_SIDE:
        scale = MAX_SIDE / max(w, h)
        img = img.resize((max(1, int(w * scale)), max(1, int(h * scale))), Image.LANCZOS)
    return img

def sha256_file(path, block_size=1024*1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(block_size), b""):
            h.update(block)
    return h.hexdigest()

def sha256_image(img):
    buf = io.BytesIO()
    coerce_image(img).save(buf, format="PNG")
    return hashlib.sha256(buf.getvalue()).hexdigest()

def _load_previous_hash():
    if os.path.exists(CHAIN_STATE_PATH):
        try:
            return json.load(open(CHAIN_STATE_PATH, encoding="utf-8")).get("last_hash", "")
        except Exception:
            return ""
    if os.path.exists(LEDGER_PATH):
        try:
            df = pd.read_csv(LEDGER_PATH)
            if len(df) and "record_hash" in df.columns:
                return str(df.iloc[-1]["record_hash"])
        except Exception:
            pass
    return ""

def _canonical_record(record):
    return json.dumps(record, ensure_ascii=False, sort_keys=True, separators=(",", ":"))

def append_ledger(record):
    previous = _load_previous_hash()
    record = dict(record)
    record["previous_record_hash"] = previous
    record["record_hash"] = hashlib.sha256(_canonical_record(record).encode("utf-8")).hexdigest()

    exists = os.path.exists(LEDGER_PATH)

    # Verifica que el encabezado del CSV coincida con LEDGER_FIELDS. La migración
    # automática (celda 2b) debería haber corregido esto antes de llegar aquí;
    # si el error aparece de todos modos, es una señal real de que hay que
    # volver a correr la migración o revisar LEDGER_FIELDS.
    if exists:
        with open(LEDGER_PATH, encoding="utf-8", newline="") as f:
            current_header = next(csv.reader(f), [])
        if current_header != LEDGER_FIELDS:
            raise RuntimeError(
                f"Esquema de bitácora desactualizado: el CSV tiene {len(current_header)} "
                f"columnas pero LEDGER_FIELDS tiene {len(LEDGER_FIELDS)}. "
                "Ejecuta la celda 2b (migración automática) antes de continuar."
            )

    with open(LEDGER_PATH, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=LEDGER_FIELDS)
        if not exists:
            writer.writeheader()
        writer.writerow({k: record.get(k, "") for k in LEDGER_FIELDS})

    with open(CHAIN_STATE_PATH, "w", encoding="utf-8") as f:
        json.dump({"last_hash": record["record_hash"], "updated_utc": utc_now()}, f, ensure_ascii=False, indent=2)

    return record


<div style="border: 1px solid #e2e8f0; border-radius: 8px; padding: 20px; background-color: #ffffff; box-shadow: 0 2px 4px rgba(0,0,0,0.02); margin: 15px 0;">
    <div style="display: flex; justify-content: space-between; align-items: center; border-bottom: 2px solid #0284c7; padding-bottom: 10px; margin-bottom: 12px;">
        <h2 style="color: #0f172a; margin: 0; font-size: 20px; border: none;">
            🎯 6. Inferencia y detección
        </h2>
        <span style="background-color: #e0f2fe; color: #0369a1; padding: 4px 10px; border-radius: 20px; font-size: 12px; font-weight: bold;">
            Fase: Localización
        </span>
    </div>
    <p style="color: #334155; margin: 0 0 10px 0; font-size: 14px;">
        <strong>Objetivo:</strong> Localizar el dispositivo en la entrada.<br>
        <strong>Descripción:</strong> Detección optimizada con YOLO.
    </p>
</div>

In [12]:
# ============================================================
# 6. INFERENCIA Y DETECCIÓN CON OPTIMIZACIÓN YOLO
# ============================================================
def detect(img, conf=YOLO_CONF):
    res = yolo(img, conf=conf, verbose=False)[0]
    boxes = []
    if res.boxes is not None and len(res.boxes) > 0:
        for b in res.boxes:
            box = b.xyxy[0].cpu().numpy().tolist()
            score = float(b.conf[0].cpu().numpy())
            cls_id = int(b.cls[0].cpu().numpy())
            label_name = res.names[cls_id]
            boxes.append((cls_id, label_name, score, [int(v) for v in box]))
    return boxes

@torch.inference_mode()
def clip_zero_shot(image, templates_dict):
    image = coerce_image(image)
    labels, prompts = [], []
    for label, plist in templates_dict.items():
        for p in plist:
            labels.append(label)
            prompts.append(p)

    image_input = biomed_preprocess(image).unsqueeze(0).to(DEVICE)
    text_tokens = biomed_tokenizer(prompts).to(DEVICE)

    image_features = biomed_model.encode_image(image_input)
    text_features = biomed_model.encode_text(text_tokens)

    image_features = image_features / image_features.norm(dim=-1, keepdim=True)
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)

    logits = image_features @ text_features.T
    try:
        logits = logits * biomed_model.logit_scale.exp()
    except Exception:
        pass

    probs = torch.softmax(logits, dim=-1).detach().float().cpu().numpy()[0]
    return probs, labels

def aggregate_by_label(probs, mapping):
    agg = {}
    for p, label in zip(probs, mapping):
        agg[label] = agg.get(label, 0.0) + float(p)
    total = sum(agg.values()) or 1.0
    return {k: v / total for k, v in agg.items()}

def entropy_metrics(agg):
    vals = np.asarray(list(agg.values()), dtype=np.float64)
    vals = vals / (vals.sum() or 1.0)
    ent = -float(np.sum(vals * np.log2(vals + 1e-12)))
    norm = ent / max(math.log2(max(len(vals), 2)), 1e-9)
    label = max(agg, key=agg.get)
    return label, float(agg[label]), float(norm)

def biomed_classify(image, templates):
    probs, mapping = clip_zero_shot(image, templates)
    agg = aggregate_by_label(probs, mapping)
    label, conf, entropy = entropy_metrics(agg)
    return {"label": label, "score": conf, "entropy": entropy, "scores": agg}

def gatekeeper(image, yolo_detections=None):
    """
    Gatekeeper: SIEMPRE valida el dominio médico con BiomedCLIP zero-shot.

    yolo_detections se recibe solo para fines informativos/localización (cajas,
    conteo de objetos) -- NO se usa para decidir el pase. YOLO aquí es el
    modelo genérico (COCO, sin fine-tuning): su score de confianza refleja qué
    tan seguro está de la clase que detectó (persona, laptop, control de
    videojuegos...), no qué tan "médica" es la imagen. Usar ese score como
    atajo de aprobación (como en una versión anterior) deja pasar objetos
    no médicos con confianza alta -- por eso la validación real recae
    siempre en BiomedCLIP, entrenado en dominio biomédico.
    """
    result = biomed_classify(image, {x: [x] for x in GATEKEEPER_TEMPLATES})
    medical_score = sum(result["scores"].get(x, 0.0) for x in MEDICAL_GATE_LABELS)
    passed = medical_score >= MEDICAL_THRESHOLD
    reason = "" if passed else f"Dominio médico insuficiente (score={medical_score:.3f}). Predicción: {result['label']}."
    return passed, medical_score, reason, result

@torch.inference_mode()
def describe_image(img, max_new_tokens=45):
    inp = blip_proc(images=coerce_image(img), return_tensors="pt")
    inp = {k: v.to(DEVICE) for k, v in inp.items()}
    if DEVICE == "cuda" and "pixel_values" in inp:
        inp["pixel_values"] = inp["pixel_values"].to(dtype=DTYPE)
    out = blip_model.generate(**inp, max_new_tokens=max_new_tokens, num_beams=3)
    return blip_proc.decode(out[0], skip_special_tokens=True).strip()

def transcribe_audio(audio_path, language="es"):
    if not audio_path or not os.path.exists(audio_path):
        return ""
    import librosa
    speech, _ = librosa.load(audio_path, sr=16000, mono=True)
    if len(speech) == 0:
        return ""
    feats = whisper_proc(speech, sampling_rate=16000, return_tensors="pt").input_features.to(DEVICE)
    if DEVICE == "cuda":
        feats = feats.to(dtype=DTYPE)
    forced = whisper_proc.get_decoder_prompt_ids(language=language, task="transcribe")
    with torch.inference_mode():
        ids = whisper_model.generate(feats, forced_decoder_ids=forced, max_new_tokens=256)
    return whisper_proc.batch_decode(ids, skip_special_tokens=True)[0].strip()

def synthesize_report(text, speaker_wav, out_path, language="es"):
    if tts is None or not speaker_wav or not os.path.exists(speaker_wav):
        return None
    try:
        tts.tts_to_file(text=text, speaker_wav=speaker_wav, language=language, file_path=out_path)
        return out_path
    except Exception as e:
        print("Error XTTS:", e)
        return None

def make_evidence_json(record, payload):
    path = os.path.join(EVIDENCE_DIR, f"{record['inspection_id']}.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2, default=str)
    return path

<div style="border: 1px solid #e2e8f0; border-radius: 8px; padding: 20px; background-color: #ffffff; box-shadow: 0 2px 4px rgba(0,0,0,0.02); margin: 15px 0;">
    <div style="display: flex; justify-content: space-between; align-items: center; border-bottom: 2px solid #0284c7; padding-bottom: 10px; margin-bottom: 12px;">
        <h2 style="color: #0f172a; margin: 0; font-size: 20px; border: none;">
            🖼️ 7. Inspección en imágenes
        </h2>
        <span style="background-color: #e0f2fe; color: #0369a1; padding: 4px 10px; border-radius: 20px; font-size: 12px; font-weight: bold;">
            Fase: Pipeline Fotos
        </span>
    </div>
    <p style="color: #334155; margin: 0 0 10px 0; font-size: 14px;">
        <strong>Objetivo:</strong> Producir una inspección por imagen.<br>
        <strong>Descripción:</strong> Pipeline completo: gatekeeper + clasificación + reporte para fotos.
    </p>
</div>

In [13]:
# ============================================================
# 7. INSPECCIÓN EN IMÁGENES (CON DECISIÓN MULTIMODAL COMPLETA)
# ============================================================
def run_satdm_inspection(image_input, voice_note_path=None, speaker_ref_path=REFERENCE_VOICE):
    current_inspection_id = inspection_id()
    img = coerce_image(image_input)
    img_hash = sha256_image(img)
    timestamp = utc_now()

    # Sanitizar la entrada de audio de Gradio (si viene como dict o tupla)
    if isinstance(voice_note_path, dict):
        voice_note_path = voice_note_path.get("name") or voice_note_path.get("path")
    elif isinstance(voice_note_path, tuple) and len(voice_note_path) > 1:
        # Si Gradio envía (sr, array), lo guardamos temporalmente en un archivo wav
        import soundfile as sf
        sr_input, audio_arr = voice_note_path
        temp_audio_path = os.path.join(OUTPUT_DIR, f"temp_{current_inspection_id}.wav")
        sf.write(temp_audio_path, audio_arr, sr_input)
        voice_note_path = temp_audio_path

    # Detección inicial con YOLO
    detections = detect(img)

    caption = describe_image(img)
    passed, gk_score, gk_reason, gk_result = gatekeeper(img, yolo_detections=detections)

    if not passed:
        record = append_ledger({
            "inspection_id": current_inspection_id, "timestamp_utc": timestamp,
            "media_type": "image", "device_id": "N/D",
            "evidence_filename": "captura.png", "evidence_sha256": img_hash,
            "gatekeeper_passed": False, "gatekeeper_score": f"{gk_score:.4f}",
            "gatekeeper_reason": gk_reason, "tipo_detectado": "", "tipo_confianza": "",
            "tipo_entropia": "", "estado_detectado": "", "estado_confianza": "",
            "estado_entropia": "", "operator_note": "", "review_required": True,
            "model_version": f"{YOLO_MODEL_SOURCE}+BiomedCLIP+BLIP+Whisper+CLAP",
            "evidence_path": "", "report_path": "", "audio_report_path": "",
            "frame_start": "", "frame_end": "", "track_id": "",
            "audio_tag_detectado": "", "audio_tag_confianza": "", "audio_tiene_voz": False,
            "resumen_reporte": f"❌ Rechazado por gatekeeper: {gk_reason}"
        })
        return f"❌ REGISTRO RECHAZADO POR GATEKEEPER: {gk_reason}", None

    type_r = biomed_classify(img, DEVICE_TYPE_TEMPLATES)
    state_r = biomed_classify(img, STATUS_TEMPLATES)

    # Análisis del audio (transcripción Whisper y evento acústico CLAP)
    audio_result = analyze_audio_note(voice_note_path)
    voice_text = audio_result["transcript"] if audio_result["has_speech"] else "Sin observaciones de voz."

    # Analizar si la nota verbal contiene palabras clave de la taxonomía de dispositivos
    voice_device_res = detect_device_from_voice(voice_text)
    voice_label = voice_device_res.get("label")
    voice_conf = voice_device_res.get("confidence", 0.0)

    # ============================================================
    # PUNTO 7: DECISIÓN FINAL Y FUSIÓN MULTIMODAL (VOZ + VISIÓN)
    # ============================================================

    # 1. Decisión del TIPO DE DISPOSITIVO
    if voice_label and voice_conf > 0.0:
        primary_device_label = voice_label
        primary_device_score = voice_conf
        origen_decision_tipo = f"Nota de Voz del Operador ('{voice_label}')"
    else:
        primary_device_label = type_r['label']
        primary_device_score = type_r['score']
        origen_decision_tipo = f"Clasificación Visual BiomedCLIP ({type_r['score']:.2%})"

    # 2. Decisión del ESTADO DEL DISPOSITIVO (revisión de palabras clave de estado en voz)
    # Cobertura ampliada: los 7 estados de STATUS_TEMPLATES ahora tienen
    # un disparador por voz equivalente (antes solo 3 de 7 eran alcanzables).
    voice_text_norm = _normalize_text_es(voice_text)
    estado_por_voz = None
    if any(k in voice_text_norm for k in ["buen estado", "funcional", "operativo"]):
        estado_por_voz = "funcional / buen estado"
    elif any(k in voice_text_norm for k in ["danado", "dano", "roto", "quebrado", "fracturado"]):
        estado_por_voz = "dañado / daño visible"
    elif any(k in voice_text_norm for k in [
        "cables expuestos", "cable expuesto", "componentes expuestos", "cableado expuesto"
    ]):
        estado_por_voz = "cables o componentes expuestos"
    elif any(k in voice_text_norm for k in ["sucio", "contaminado", "manchado", "con residuos"]):
        estado_por_voz = "sucio o contaminado visible"
    elif any(k in voice_text_norm for k in [
        "corrosion", "oxidacion", "oxidado", "oxidada", "corroido", "corroida"
    ]):
        estado_por_voz = "corrosión u oxidación visible"
    elif any(k in voice_text_norm for k in [
        "etiqueta ilegible", "numero de serie ilegible", "etiqueta borrada",
        "serie borrado", "sin etiqueta legible"
    ]):
        estado_por_voz = "etiqueta o número de serie ilegible"
    elif any(k in voice_text_norm for k in [
        "no se puede determinar", "no concluyente", "no es claro el estado", "estado incierto"
    ]):
        estado_por_voz = "estado no concluyente"

    if estado_por_voz:
        primary_state_label = estado_por_voz
        # Confianza calibrada (no 1.0) para que la entropía visual siga
        # pudiendo disparar revisión manual aunque la voz haya decidido la etiqueta.
        primary_state_score = VOICE_CONF_OVERRIDE
        origen_decision_estado = f"Nota de Voz del Operador ('{estado_por_voz}')"
    else:
        primary_state_label = state_r['label']
        primary_state_score = state_r['score']
        origen_decision_estado = f"Clasificación Visual BiomedCLIP ({state_r['score']:.2%})"

    report_text = (
        f"Inspección SATDM #{current_inspection_id}.\n"
        f"Tipo de Dispositivo Detectado: {primary_device_label} (Vía: {origen_decision_tipo}).\n"
        f"Estado Detectado: {primary_state_label} (Vía: {origen_decision_estado}, Entropía: {state_r['entropy']:.2f}).\n"
        f"Evento Acústico: {audio_result['audio_tag'] or 'N/D'} (Confianza: {audio_result['audio_tag_score']:.2%}).\n"
        f"Nota de Voz del Técnico: {voice_text}\n"
        f"Descripción Visual (BLIP): {caption}"
    )

    review_required = (
        primary_device_score < MEDICAL_THRESHOLD or
        primary_state_score < MEDICAL_THRESHOLD or
        type_r["entropy"] > AMBIGUITY_ENTROPY or
        state_r["entropy"] > AMBIGUITY_ENTROPY
    )
    if review_required:
        report_text += "\n⚠️ Revisión manual recomendada."

    output_audio_path = None
    if ENABLE_TTS and tts is not None and os.path.exists(speaker_ref_path or ""):
        output_audio_path = os.path.join(OUTPUT_DIR, f"{current_inspection_id}_report.wav")
        output_audio_path = synthesize_report(report_text, speaker_ref_path, output_audio_path)

    device_id_hash = hashlib.sha256(primary_device_label.encode('utf-8')).hexdigest()[:12]

    evidence_json_path = os.path.join(EVIDENCE_DIR, f"{current_inspection_id}.json")

    record = append_ledger({
        "inspection_id": current_inspection_id, "timestamp_utc": timestamp,
        "media_type": "image", "device_id": device_id_hash,
        "evidence_filename": "captura.png", "evidence_sha256": img_hash,
        "gatekeeper_passed": True, "gatekeeper_score": f"{gk_score:.4f}",
        "gatekeeper_reason": "", "tipo_detectado": primary_device_label,
        "tipo_confianza": f"{primary_device_score:.4f}", "tipo_entropia": f"{type_r['entropy']:.4f}",
        "estado_detectado": primary_state_label, "estado_confianza": f"{primary_state_score:.4f}",
        "estado_entropia": f"{state_r['entropy']:.4f}", "operator_note": voice_text,
        "review_required": review_required,
        "model_version": f"{YOLO_MODEL_SOURCE}+BiomedCLIP+BLIP+Whisper+CLAP",
        "evidence_path": evidence_json_path, "report_path": "", "audio_report_path": output_audio_path or "",
        "frame_start": "", "frame_end": "", "track_id": "",
        "audio_tag_detectado": audio_result["audio_tag"],
        "audio_tag_confianza": f"{audio_result['audio_tag_score']:.4f}",
        "audio_tiene_voz": audio_result["has_speech"],
        "resumen_reporte": report_text
    })

    make_evidence_json(record, {
        "gatekeeper": gk_result,
        "yolo_detections": detections,
        "visual_type": type_r,
        "voice_device": voice_device_res,
        "decision_source_type": origen_decision_tipo,
        "decision_source_state": origen_decision_estado,
        "state": state_r,
        "caption": caption,
        "audio": audio_result
    })

    return report_text, output_audio_path

<div style="border: 1px solid #e2e8f0; border-radius: 8px; padding: 20px; background-color: #ffffff; box-shadow: 0 2px 4px rgba(0,0,0,0.02); margin: 15px 0;">
    <div style="display: flex; justify-content: space-between; align-items: center; border-bottom: 2px solid #0284c7; padding-bottom: 10px; margin-bottom: 12px;">
        <h2 style="color: #0f172a; margin: 0; font-size: 20px; border: none;">
            🎥 8. Inspección en video
        </h2>
        <span style="background-color: #e0f2fe; color: #0369a1; padding: 4px 10px; border-radius: 20px; font-size: 12px; font-weight: bold;">
            Fase: Pipeline Video
        </span>
    </div>
    <p style="color: #334155; margin: 0 0 10px 0; font-size: 14px;">
        <strong>Objetivo:</strong> Cubrir el caso de uso de inspección en video.<br>
        <strong>Descripción:</strong> Extiende el pipeline a clips de video.
    </p>
</div>

In [14]:
# ============================================================
# 8. INSPECCIÓN EN VIDEO
# ============================================================
def inspect_video(
    video_path, output_path=None, frame_stride=VIDEO_FRAME_STRIDE,
    conf=YOLO_CONF, tracker="bytetrack.yaml", device_id="", audio_note=None
):
    if not video_path or not os.path.exists(video_path):
        raise ValueError("Video no encontrado.")

    current_id = inspection_id()
    output_path = output_path or os.path.join(OUTPUT_DIR, f"{Path(video_path).stem}_SATDM.mp4")
    video_hash = sha256_file(video_path)

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise ValueError("No se pudo abrir el archivo de video.")

    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH) or 0)
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or 0)
    cap.release()

    writer = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (width, height))
    memory = {}
    frame_index = 0
    stride = max(1, int(frame_stride))

    try:
        results = yolo.track(source=video_path, stream=True, persist=True, tracker=tracker, conf=conf, verbose=False)
        for result in results:
            frame = result.orig_img.copy()
            boxes = result.boxes

            if boxes is not None and len(boxes) > 0 and boxes.id is not None:
                ids = boxes.id.int().cpu().tolist()
                xyxy = boxes.xyxy.int().cpu().tolist()
                scores = boxes.conf.cpu().tolist()
                clss = boxes.cls.int().cpu().tolist()

                for tid, box, det_score, cls_id in zip(ids, xyxy, scores, clss):
                    x1, y1, x2, y2 = [max(0, v) for v in box]
                    crop = frame[y1:y2, x1:x2]
                    if crop.size == 0:
                        continue

                    if tid not in memory:
                        memory[tid] = {
                            "type_scores": {}, "state_scores": {}, "observations": 0,
                            "first_frame": frame_index, "last_frame": frame_index,
                            "last_classified_frame": -stride, "class_name": result.names[cls_id]
                        }

                    data = memory[tid]
                    data["last_frame"] = frame_index

                    if frame_index - data["last_classified_frame"] >= stride:
                        crop_pil = Image.fromarray(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
                        gk_pass, _, _, _ = gatekeeper(crop_pil)
                        if gk_pass:
                            tr = biomed_classify(crop_pil, DEVICE_TYPE_TEMPLATES)
                            sr = biomed_classify(crop_pil, STATUS_TEMPLATES)

                            for k, v in tr["scores"].items():
                                data["type_scores"][k] = data["type_scores"].get(k, 0.0) + float(v)
                            for k, v in sr["scores"].items():
                                data["state_scores"][k] = data["state_scores"].get(k, 0.0) + float(v)
                            data["observations"] += 1

                        data["last_classified_frame"] = frame_index

                    label = data["class_name"]
                    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                    cv2.putText(frame, f"ID:{tid} {label}", (x1, max(15, y1 - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

            writer.write(frame)
            frame_index += 1
    finally:
        writer.release()

    operator_note = transcribe_audio(audio_note) if audio_note else "Sin nota de voz."

    report_summary = [f"Inspección de Video #{current_id}", f"Frames: {frame_index}, Objetos Rastreados: {len(memory)}"]
    for tid, data in memory.items():
        if data["type_scores"]:
            t_label, t_conf, t_entropy = entropy_metrics(data["type_scores"])
            s_label, s_conf, s_entropy = entropy_metrics(data["state_scores"])
        else:
            # Sin observaciones BiomedCLIP para este track: no hay evidencia real
            # de tipo/estado, así que se fuerza entropía máxima para garantizar
            # revisión manual en vez de asumir silenciosamente que está bien.
            t_label, t_conf, t_entropy = data["class_name"], 1.0, 1.0
            s_label, s_conf, s_entropy = "Indeterminado", 0.0, 1.0

        track_summary = f"Track {tid} ({data['class_name']}): {t_label} ({t_conf:.1%}) | Estado: {s_label} ({s_conf:.1%})"
        report_summary.append(f"- {track_summary}")

        # Misma rigurosidad que en el pipeline de imagen: confianza Y entropía
        # de tipo Y estado, no solo la confianza de tipo.
        track_review_required = (
            t_conf < MEDICAL_THRESHOLD or
            s_conf < MEDICAL_THRESHOLD or
            t_entropy > AMBIGUITY_ENTROPY or
            s_entropy > AMBIGUITY_ENTROPY
        )

        append_ledger({
            "inspection_id": current_id, "timestamp_utc": utc_now(), "media_type": "video",
            "device_id": device_id or f"TRACK-{tid}", "evidence_filename": os.path.basename(video_path),
            "evidence_sha256": video_hash, "gatekeeper_passed": True, "gatekeeper_score": "1.0",
            # Antes se guardaba data["class_name"] (la clase genérica de YOLO/COCO,
            # ej. "person", "laptop") en vez de t_label (el tipo de dispositivo real
            # identificado por BiomedCLIP). Corregido para que la bitácora refleje
            # lo mismo que reporta track_summary.
            "tipo_detectado": t_label, "tipo_confianza": f"{t_conf:.4f}",
            "tipo_entropia": f"{t_entropy:.4f}",
            "estado_detectado": s_label, "estado_confianza": f"{s_conf:.4f}",
            "estado_entropia": f"{s_entropy:.4f}",
            "operator_note": operator_note, "review_required": track_review_required,
            "evidence_path": output_path, "frame_start": data["first_frame"],
            "frame_end": data["last_frame"], "track_id": tid,
            "resumen_reporte": f"{track_summary} | Voz: {operator_note}"
        })

    return "\n".join(report_summary), output_path

<div style="border: 1px solid #e2e8f0; border-radius: 8px; padding: 20px; background-color: #ffffff; box-shadow: 0 2px 4px rgba(0,0,0,0.02); margin: 15px 0;">
    <div style="display: flex; justify-content: space-between; align-items: center; border-bottom: 2px solid #0284c7; padding-bottom: 10px; margin-bottom: 12px;">
        <h2 style="color: #0f172a; margin: 0; font-size: 20px; border: none;">
            📱 9. Interfaz Gradio
        </h2>
        <span style="background-color: #e0f2fe; color: #0369a1; padding: 4px 10px; border-radius: 20px; font-size: 12px; font-weight: bold;">
            Fase: Despliegue
        </span>
    </div>
    <p style="color: #334155; margin: 0 0 10px 0; font-size: 14px;">
        <strong>Objetivo:</strong> Uso en campo por personal de salud.<br>
        <strong>Descripción:</strong> App completa con bitácora visible, manos-libres.
    </p>
</div>

In [15]:
# ============================================================
# 8b. PREVUE / AUTODIAGNÓSTICO SATDM
# ============================================================
def satdm_preflight():
    checks = {
        "Drive / raíz": os.path.exists(DRIVE_ROOT),
        "Directorio de salida": os.path.exists(OUTPUT_DIR),
        "Bitácora": os.path.exists(LEDGER_PATH),
        "YOLO": "yolo" in globals(),
        "BiomedCLIP": "biomed_model" in globals(),
        "BLIP": "blip_model" in globals(),
        "Whisper": "whisper_model" in globals(),
        "CLAP": bool(globals().get("CLAP_AVAILABLE", False)),
    }

    print("=" * 70)
    print("SATDM — AUTODIAGNÓSTICO")
    print("=" * 70)
    for name, ok in checks.items():
        print(f"{'OK' if ok else 'WARN'}  {name}")

    print("-" * 70)
    print("YOLO:", globals().get("YOLO_MODEL_SOURCE", "No disponible"))
    print("CLAP:", "Disponible" if checks["CLAP"] else "No disponible")
    print("Dispositivo:", DEVICE)
    print("Salida:", OUTPUT_DIR)
    print("Bitácora:", LEDGER_PATH)
    print("=" * 70)

    return checks

SATDM_PREFLIGHT = satdm_preflight()


SATDM — AUTODIAGNÓSTICO
OK  Drive / raíz
OK  Directorio de salida
OK  Bitácora
OK  YOLO
OK  BiomedCLIP
OK  BLIP
OK  Whisper
OK  CLAP
----------------------------------------------------------------------
YOLO: pesos_medicos_yolov8.pt
CLAP: Disponible
Dispositivo: cpu
Salida: /content/drive/MyDrive/TAE_IA_M6/SATDM_output
Bitácora: /content/drive/MyDrive/TAE_IA_M6/SATDM_ledger/inspecciones_log.csv


In [16]:

# ============================================================
# 9. INTERFAZ COMPLETA DE LA APP SATDM
# Sistema Auditable de Trazabilidad de Dispositivos Médicos
# ============================================================

import os
import pandas as pd
import gradio as gr
from datetime import datetime, timezone

# ------------------------------------------------------------
# CONFIGURACIÓN
# ------------------------------------------------------------

APP_TITLE = "SATDM — Inspector de Dispositivos Médicos"

APP_DESCRIPTION = """
Sistema Auditable de Trazabilidad de Dispositivos Médicos.
Inspección mediante visión artificial, voz y bitácora auditable.
"""

# ------------------------------------------------------------
# FUNCIONES AUXILIARES DE LA INTERFAZ
# ------------------------------------------------------------

def cargar_bitacora():
    """
    Carga los últimos registros de la bitácora SATDM.
    """
    try:
        if os.path.exists(LEDGER_PATH):
            df = pd.read_csv(LEDGER_PATH, on_bad_lines="skip")  # tolera filas corruptas residuales
            if len(df) > 0:
                return df.tail(20)
        return pd.DataFrame(columns=LEDGER_FIELDS)  # usa el esquema real, no una lista aparte
    except Exception as e:
        print("Error leyendo bitácora:", e)
        return pd.DataFrame(columns=LEDGER_FIELDS)




def ejecutar_inspeccion(imagen, audio_nota):
    if imagen is None:
        bitacora = cargar_bitacora()
        return (
            "No se recibió ninguna imagen.",
            None,
            bitacora,
            bitacora
        )

    try:
        # Aseguramos que la nota de voz se envíe explícitamente a voice_note_path
        reporte, audio_salida = run_satdm_inspection(
            image_input=imagen,
            voice_note_path=audio_nota
        )

        bitacora = cargar_bitacora()
        return (
            reporte,
            audio_salida,
            bitacora,
            bitacora
        )

    except Exception as e:
        error = (
            "ERROR DURANTE LA INSPECCIÓN SATDM\n\n"
            f"{type(e).__name__}: {str(e)}"
        )
        bitacora = cargar_bitacora()
        return (
            error,
            None,
            bitacora,
            bitacora
        )
    try:

        reporte, audio_salida = run_satdm_inspection(
            imagen,
            audio_nota
        )

        bitacora = cargar_bitacora()

        return (
            reporte,
            audio_salida,
            bitacora,
            bitacora
        )

    except Exception as e:

        error = (
            "ERROR DURANTE LA INSPECCIÓN SATDM\n\n"
            f"{type(e).__name__}: {str(e)}"
        )
        bitacora = cargar_bitacora()

        return (
            error,
            None,
            bitacora,
            bitacora
        )


def ejecutar_video(
    video,
    nota_voz,
    device_id,
    frame_stride
):
    """
    Ejecuta la inspección de video.
    """

    if video is None:
        bitacora = cargar_bitacora()
        return (
            "No se recibió ningún video.",
            None,
            bitacora,
            bitacora
        )

    try:

        video_path = video

        output_name = (
            f"{os.path.splitext(os.path.basename(video_path))[0]}"
            "_SATDM.mp4"
        )

        output_path = os.path.join(
            OUTPUT_DIR,
            output_name
        )

        reporte, video_salida = inspect_video(
            video_path=video_path,
            output_path=output_path,
            frame_stride=int(frame_stride),
            device_id=device_id or "",
            audio_note=nota_voz
        )

        bitacora = cargar_bitacora()

        return (
            reporte,
            video_salida,
            bitacora,
            bitacora
        )

    except Exception as e:

        error = (
            "ERROR DURANTE LA INSPECCIÓN DE VIDEO\n\n"
            f"{type(e).__name__}: {str(e)}"
        )
        bitacora = cargar_bitacora()

        return (
            error,
            None,
            bitacora,
            bitacora
        )


def actualizar_bitacora():
    """
    Actualiza manualmente la tabla de auditoría.
    """
    return cargar_bitacora()


def limpiar_imagen():
    return None


def limpiar_video():
    return None


def informacion_sistema():

    modelo_yolo = "yolov8n.pt (COCO, sin fine-tuning)"

    try:
        dispositivo = DEVICE
    except Exception:
        dispositivo = "No disponible"

    try:
        ledger = LEDGER_PATH
    except Exception:
        ledger = "No disponible"

    texto = f"""
SATDM — INFORMACIÓN DEL SISTEMA

Fecha UTC:
{datetime.now(timezone.utc).isoformat()}

Dispositivo de cómputo:
{dispositivo}

Modelo YOLO:
{modelo_yolo}

Archivo de bitácora:
{ledger}

Componentes del sistema:

• YOLO — detección y localización; usa pesos médicos si están disponibles
• Gatekeeper — validación del dominio médico
• BiomedCLIP — clasificación
• BLIP — descripción visual
• Whisper — transcripción de voz
• XTTS — generación opcional de audio
• SHA-256 — integridad de evidencia
• CLAP — análisis acústico zero-shot
• Ledger SATDM — trazabilidad de inspecciones
"""

    return texto


# ============================================================
# CONSTRUCCIÓN DE LA APLICACIÓN
# ============================================================

with gr.Blocks(
    title=APP_TITLE,
    theme=gr.themes.Soft()
) as app:

    # --------------------------------------------------------
    # ENCABEZADO
    # --------------------------------------------------------

    gr.Markdown(
        """
        # SATDM
        ## Inspector de Dispositivos Médicos

        **Sistema Auditable de Trazabilidad de Dispositivos Médicos**

        Inspección, clasificación, evidencia y trazabilidad.

        **IA integrada:** YOLO médico (si los pesos están disponibles),
        BiomedCLIP, BLIP, Whisper y CLAP.
        """
    )

    gr.Markdown(
        """
        ---
        **Flujo SATDM:**

        Evidencia → Detección → Validación → Clasificación →
        Estado → Reporte → Bitácora auditable
        ---
        """
    )

    # ========================================================
    # PESTAÑA 1 — INSPECCIÓN DE IMAGEN
    # ========================================================

    with gr.Tab("Inspección de Imagen"):

        gr.Markdown(
            """
            ### Inspección de dispositivo médico

            Capture o cargue una fotografía del dispositivo.
            Opcionalmente puede agregarse una nota de voz del técnico.
            """
        )

        with gr.Row():

            # ------------------------------------------------
            # ENTRADAS
            # ------------------------------------------------

            with gr.Column(scale=1):

                imagen_input = gr.Image(
                    type="pil",
                    label="Fotografía del dispositivo médico"
                )

                audio_input = gr.Audio(
                    sources=[
                        "microphone",
                        "upload"
                    ],
                    type="filepath",
                    label="Nota de voz del técnico — opcional"
                )

                with gr.Row():

                    boton_inspeccionar = gr.Button(
                        "Ejecutar inspección",
                        variant="primary"
                    )

                    boton_limpiar = gr.Button(
                        "Limpiar"
                    )

            # ------------------------------------------------
            # RESULTADOS
            # ------------------------------------------------

            with gr.Column(scale=1):

                reporte_imagen = gr.Textbox(
                    label="Reporte SATDM",
                    lines=12,
                    interactive=False
                )

                audio_reporte = gr.Audio(
                    label="Reporte narrado",
                    type="filepath",
                    interactive=False
                )

        gr.Markdown(
            "### Resultado de la inspección"
        )

        tabla_imagen = gr.Dataframe(
            label="Últimos registros SATDM",
            interactive=False
        )

    # ========================================================
    # PESTAÑA 2 — INSPECCIÓN DE VIDEO
    # ========================================================

    with gr.Tab("Inspección de Video"):

        gr.Markdown(
            """
            ### Inspección de dispositivos mediante video

            El sistema utiliza seguimiento de objetos y clasificación
            para mantener la identidad de cada dispositivo detectado.
            """
        )

        with gr.Row():

            with gr.Column(scale=1):

                video_input = gr.Video(
                    label="Video del dispositivo médico"
                )

                video_device_id = gr.Textbox(
                    label="ID del dispositivo",
                    placeholder="Ejemplo: SATDM-DEV-001"
                )

                video_audio = gr.Audio(
                    sources=[
                        "microphone",
                        "upload"
                    ],
                    type="filepath",
                    label="Nota de voz del técnico — opcional"
                )

                video_stride = gr.Slider(
                    minimum=1,
                    maximum=60,
                    value=10,
                    step=1,
                    label="Intervalo de clasificación de frames"
                )

                with gr.Row():

                    boton_video = gr.Button(
                        "Ejecutar inspección de video",
                        variant="primary"
                    )

                    boton_video_limpiar = gr.Button(
                        "Limpiar"
                    )

            with gr.Column(scale=1):

                reporte_video = gr.Textbox(
                    label="Reporte de video SATDM",
                    lines=15,
                    interactive=False
                )

                video_salida = gr.Video(
                    label="Video procesado",
                    interactive=False
                )

        gr.Markdown(
            "### Registros generados"
        )

        tabla_video = gr.Dataframe(
            label="Bitácora SATDM",
            interactive=False
        )

    # ========================================================
    # PESTAÑA 3 — BITÁCORA
    # ========================================================

    with gr.Tab("Bitácora Auditable"):

        gr.Markdown(
            """
            ### Bitácora SATDM

            Registros generados por las inspecciones realizadas.
            """
        )

        boton_actualizar = gr.Button(
            "Actualizar bitácora"
        )

        tabla_ledger = gr.Dataframe(
            value=cargar_bitacora(),
            label="Últimas inspecciones",
            interactive=False
        )

        gr.Markdown(
            """
            La bitácora puede contener información como:

            - Identificador de inspección
            - Fecha y hora UTC
            - Tipo de evidencia
            - Identificador del dispositivo
            - Dispositivo detectado
            - Confianza de clasificación
            - Estado detectado
            - Confianza del estado
            - SHA-256 de la evidencia
            - Revisión manual requerida
            """
        )

    # ========================================================
    # PESTAÑA 4 — INFORMACIÓN DEL SISTEMA
    # ========================================================

    with gr.Tab("Sistema"):

        gr.Markdown(
            """
            ### Estado del sistema SATDM
            """
        )

        boton_info = gr.Button(
            "Consultar configuración"
        )

        info_sistema = gr.Textbox(
            label="Configuración actual",
            lines=20,
            interactive=False
        )

        boton_info.click(
            fn=informacion_sistema,
            inputs=[],
            outputs=info_sistema
        )

    # ========================================================
    # PESTAÑA 5 — AYUDA
    # ========================================================

    with gr.Tab("Ayuda"):

        gr.Markdown(
            """
            # Guía rápida de operación

            ## 1. Inspección de imagen

            1. Cargue una fotografía.
            2. Agregue una nota de voz si es necesario.
            3. Presione **Ejecutar inspección**.
            4. Revise el reporte.
            5. Consulte el registro generado en la bitácora.

            ## 2. Inspección de video

            1. Cargue el video.
            2. Introduzca el ID del dispositivo si está disponible.
            3. Seleccione el intervalo de clasificación.
            4. Presione **Ejecutar inspección de video**.
            5. Revise el video procesado y el reporte.

            ## 3. Bitácora

            La pestaña **Bitácora Auditable** permite visualizar
            los registros generados por el sistema.

            ## 4. Revisión manual

            Cuando la clasificación presente confianza insuficiente
            o ambigüedad, el sistema puede marcar el registro para
            revisión manual.

            ## 5. Evidencia

            Cada inspección puede asociarse con evidencia y metadatos
            destinados a mantener la trazabilidad del proceso.
            """
        )


    # ========================================================
    # EVENTOS — IMAGEN
    # ========================================================

    boton_inspeccionar.click(
        fn=ejecutar_inspeccion,
        inputs=[
            imagen_input,
            audio_input
        ],
        outputs=[
            reporte_imagen,
            audio_reporte,
            tabla_imagen,
            tabla_ledger
        ]
    )

    boton_limpiar.click(
        fn=limpiar_imagen,
        inputs=[],
        outputs=imagen_input
    )


    # ========================================================
    # EVENTOS — VIDEO
    # ========================================================

    boton_video.click(
        fn=ejecutar_video,
        inputs=[
            video_input,
            video_audio,
            video_device_id,
            video_stride
        ],
        outputs=[
            reporte_video,
            video_salida,
            tabla_video,
            tabla_ledger
        ]
    )

    boton_video_limpiar.click(
        fn=limpiar_video,
        inputs=[],
        outputs=video_input
    )


    # ========================================================
    # EVENTOS — BITÁCORA
    # ========================================================

    boton_actualizar.click(
        fn=actualizar_bitacora,
        inputs=[],
        outputs=tabla_ledger
    )


# ============================================================
# FINAL
# ============================================================

print("=" * 70)
print("SATDM — INTERFAZ LISTA")
print("=" * 70)
print()
print("Para iniciar la aplicación ejecute:")
print()
print("    app.launch(share=True, debug=True)")
print()
print("=" * 70)



SATDM — INTERFAZ LISTA

Para iniciar la aplicación ejecute:

    app.launch(share=True, debug=True)



In [17]:
app.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://7e73545fbcbb4950ad.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://7e73545fbcbb4950ad.gradio.live
